# 04 - Classifier Training Across Configurations A, B, and C

Trains LR / RF / XGBoost / SVM / ANN on three feature configurations, runs 10-fold CV, and writes Tables 4.3 - 4.7 to `data/processed/`.

In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path('.').resolve().parent))
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

from code import config, preprocess, evaluate, classifiers, embeddings, fusion

merged = pd.read_parquet(config.PROCESSED_DIR / 'bank_merged.parquet')
y = merged[config.TARGET_COL].astype(int).values

# Configuration A - traditional
X_A_df, names_A = preprocess.encode_traditional(merged.drop(columns=[config.TARGET_COL]))
X_A = X_A_df.values

# Configuration B - LLM embeddings only
X_B = embeddings.load_cached('bank_emb_st')
names_B = [f'st_emb_{i}' for i in range(X_B.shape[1])]

# Configuration C - Hybrid
X_C = np.concatenate([X_A, X_B], axis=1)
names_C = list(names_A) + names_B
# add complaint features (already in X_A from join_to_bank)
print('A:', X_A.shape, 'B:', X_B.shape, 'C:', X_C.shape)

In [ ]:
rows = []
for cfg, X, names in [('A', X_A, names_A), ('B', X_B, names_B), ('C', X_C, names_C)]:
    for clf in classifiers.CLASSIFIERS:
        pipe = classifiers.make_pipeline(clf)
        cv = evaluate.cross_validate_metrics(pipe, X, y)
        for metric in evaluate.METRICS:
            if metric in cv.columns:
                rows.append({
                    'config': cfg, 'clf': clf, 'metric': metric,
                    'mean': float(cv[metric].mean()), 'std': float(cv[metric].std(ddof=1))
                })
        print(cfg, clf, 'F1=%.3f +/- %.3f' % (cv['f1'].mean(), cv['f1'].std(ddof=1)))
results = pd.DataFrame(rows)
results.to_csv(config.PROCESSED_DIR / 'cv_results.csv', index=False)

In [ ]:
# Build Tables 4.3-4.5 in markdown form
for cfg in ('A', 'B', 'C'):
    sub = results[results.config == cfg].pivot_table(index='clf', columns='metric', values='mean')
    sub.to_csv(config.PROCESSED_DIR / f'table_4_cfg_{cfg}.csv')
    print(f'\nConfiguration {cfg}'); print(sub.round(3))

In [ ]:
# Per-fold scores for the best model in each configuration -> paired t-test
best_per_cfg = results[results.metric == 'f1'].sort_values('mean', ascending=False).groupby('config').head(1)
print(best_per_cfg)

# Re-run CV for the best models (cheap because hyper-parameters fixed)
fold_f1 = {}
for _, row in best_per_cfg.iterrows():
    cfg = row.config
    X = {'A': X_A, 'B': X_B, 'C': X_C}[cfg]
    pipe = classifiers.make_pipeline(row.clf)
    cv = evaluate.cross_validate_metrics(pipe, X, y)
    fold_f1[cfg] = cv['f1'].values

from itertools import combinations
stat_rows = []
for a, b in combinations(['A', 'B', 'C'], 2):
    t, p = evaluate.paired_t_test(fold_f1[b], fold_f1[a])
    stat_rows.append({'comparison': f'{b} vs {a}', 't': t, 'p': p})
stat_df = pd.DataFrame(stat_rows)
stat_df.to_csv(config.PROCESSED_DIR / 'table_4_7_significance.csv', index=False)
stat_df

In [ ]:
# Persist the best Hybrid (C) XGBoost model for SHAP and retention generation
import joblib
best_C = classifiers.make_pipeline('xgb')
X_train, X_test, y_train, y_test = train_test_split(
    X_C, y, test_size=config.TEST_SIZE, stratify=y, random_state=config.PRIMARY_SEED)
best_C.fit(X_train, y_train)
joblib.dump({'pipe': best_C, 'feature_names': names_C, 'X_test': X_test, 'y_test': y_test},
            config.PROCESSED_DIR / 'best_hybrid_xgb.joblib')
print('Saved best Hybrid pipeline.')